# WH-M02: Scala/Spark PostgreSQL-to-Hive Silver Pipeline

| What to expect | Value |
| --- | --- |
| Scenario | Retail sales warehousing |
| Difficulty | Medium |
| Delivery scope | Pipeline slice |
| Expected effort | Two sessions, 90–150 minutes each |
| Deliverable | A rerunnable Scala/Spark Bronze-to-Silver job |
| Primary interface | Scala Spark DataFrame API |
| Prerequisites | Scala foundations, Spark DataFrames, PostgreSQL, Hive basics |
| Tooling | `scala`, `spark`, `sbt`, `postgresql`, `hive`, `parquet`, `docker` |
| Techniques | `jdbc-read`, `schema-contract`, `left-anti-join`, `deduplication`, `validation`, `quarantine`, `idempotent-rerun` |

This lab uses a Python notebook as the guide, runner, and feedback surface. Your actual pipeline work remains in the Scala/sbt project at `/Users/richardwilkerson/IdeaProjects/HelloWorld/src/main/scala/WeekendHomework.scala`.

## Start here

1. Open `WeekendHomework.scala` in IntelliJ.
2. Keep this notebook open in VS Code with the BigDataExample `.venv` Python kernel.
3. Complete Part 1 and stop at its checkpoint before starting transformations.
4. Complete Part 2 only after the setup-only run proves that connections and schema contracts work.
5. The [solution explanation](../../../docs/interactive-data-engineering-labs/solutions/retail-sales/wh-m02-scala-spark-postgres-to-hive.md) is a spoiler; open it after attempting the checkpoints.


## Pipeline and ownership map

```text
PostgreSQL container
week3_hive_migration.bronze.sales_raw
              |
              | JDBC through host port 5432
              v
Scala/Spark driver on the Mac
              |
              | Part 2 DataFrame transformations
              v
temporary Parquet on the Mac
              |
              | docker cp
              v
Hive container warehouse + HiveServer2 metadata
              |
              v
silver.sales_clean and silver.sales_rejected
```

The durable questions are: What is authoritative? What is one row? Which process can see which network and filesystem? What schema crosses each boundary? What evidence proves the boundary worked?


## Checkpoint 0: Locate the repository, Scala project, and checker

<details>
<summary>Open environment instructions and job connection</summary>

The notebook coordinates commands, but it does not contain the Scala solution. The next cell locates the BigDataExample repository, the external HelloWorld project, `WeekendHomework.scala`, the packaged JAR, and this lab's learner-facing checker.

Set `HELLO_WORLD_PROJECT` before starting the notebook kernel if the Scala project moves to another location. Do not place passwords in notebook cells. The Scala submission loads the project-local `.env` in its own shell.

**Acceptance evidence:** Every required path prints and exists.

**Job connection:** A notebook, source repository, build artifact, runtime launcher, configuration file, and stateful services have different lifecycles. A reproducible job names each boundary explicitly.

</details>


In [ ]:
from pathlib import Path
import os
import re
import shlex
import subprocess
import sys

current_path = Path.cwd().resolve()
project_root = next(
    candidate
    for candidate in (current_path, *current_path.parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "AGENTS.md").is_file()
)
scala_project = Path(
    os.environ.get("HELLO_WORLD_PROJECT", "/Users/richardwilkerson/IdeaProjects/HelloWorld")
).expanduser().resolve()
scala_source = scala_project / "src/main/scala/WeekendHomework.scala"
spark_jar = scala_project / "target/out/jvm/scala-2.13.8/helloworld/helloworld_2.13-0.1.0-SNAPSHOT.jar"
verify_script = project_root / "scripts/interactive-data-engineering-labs/retail-sales/wh-m02-scala-spark-postgres-to-hive/verify-lab.py"

def run_command(command, *, cwd=project_root):
    result = subprocess.run(
        [str(part) for part in command],
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    return result

def run_scala_shell(command):
    return run_command(
        ["/bin/zsh", "-lc", f"source .env; {command}"],
        cwd=scala_project,
    )

for required_path in (scala_project, scala_source, scala_project / "build.sbt", scala_project / ".env", verify_script):
    assert required_path.exists(), f"Missing required path: {required_path}"

print(f"Repository: {project_root}")
print(f"Scala project: {scala_project}")
print(f"Learner source: {scala_source}")
print(f"Expected JAR: {spark_jar}")


# Part 1: Setup, connections, and schema contracts

Part 1 may idempotently create the Hive `silver` namespace, clean-table directory, and external-table metadata. It must not clean, reject, transform, or append rows.

The word **schema** has three meanings here:

| Boundary | Meaning | Example |
| --- | --- | --- |
| PostgreSQL schema | Namespace inside one database | `week3_hive_migration.bronze.sales_raw` |
| Hive database | Namespace in the Hive metastore | `silver.sales_clean` |
| Spark `StructType` | Ordered DataFrame column/type/nullability contract | `silverOutputSchema` |

Creating one does not automatically create the others.


## Checkpoint 1: Start and verify PostgreSQL and Hive

<details>
<summary>Open service instructions, acceptance evidence, and job connection</summary>

The next cell starts the repository PostgreSQL Compose service and the existing `hive-server` container, then invokes the learner-facing infrastructure check. Starting a container proves only that its process launched; the checker also verifies the Bronze table and HiveServer2 through the actual database interfaces. If `silver.sales_clean` already exists, the checker validates its schema; otherwise Part 1 will create its directory and metadata.

**Acceptance evidence:** The checker prints `PASS`, a positive PostgreSQL Bronze count, and whether clean Silver already exists or still needs setup.

**Job connection:** Process health, network reachability, catalog presence, table schema, and data readability are separate operational signals.

</details>


In [ ]:
postgres_start = run_command([
    "docker", "compose",
    "--env-file", "infra/postgres/.env",
    "-f", "infra/postgres/compose.yml",
    "up", "-d",
])
assert postgres_start.returncode == 0

hive_start = run_command(["docker", "start", "hive-server"])
assert hive_start.returncode == 0

infrastructure_check = run_command([sys.executable, verify_script, "--mode", "infrastructure"])
assert infrastructure_check.returncode == 0


## Checkpoint 2: Inspect both systems independently

<details>
<summary>Open inspection commands and questions</summary>

PostgreSQL is reached from host Spark through `127.0.0.1:5432`. Hive is different: Beeline runs inside `hive-server`, and Parquet crosses the Mac/container filesystem boundary with `docker cp`.

The next cell proves the selected PostgreSQL database, displays the Bronze source definition and count, and lists Hive Silver tables. It describes and counts `sales_clean` only when that table already exists.

Ask: Which command verifies a process? Which verifies a namespace? Which verifies column types? Which verifies readable rows?

**Acceptance evidence:** PostgreSQL reports `week3_hive_migration`; HiveServer2 returns its catalog. If `sales_clean` exists, its nine columns match `silverOutputSchema`; otherwise record that Part 1 must create it.

</details>


In [ ]:
run_command([
    "docker", "exec", "big-data-example-postgres",
    "psql", "-X", "-U", "bigdata", "-d", "week3_hive_migration",
    "-c", r"\conninfo",
    "-c", r"\d bronze.sales_raw",
    "-c", "SELECT COUNT(*) AS bronze_rows FROM bronze.sales_raw;",
])

hive_tables = run_command([
    "docker", "exec", "hive-server", "beeline",
    "-u", "jdbc:hive2://localhost:10000/default",
    "--silent=true", "--showHeader=true", "--outputformat=tsv2",
    "-e", "SHOW DATABASES; SHOW TABLES IN silver;",
])
assert hive_tables.returncode == 0
if "sales_clean" in hive_tables.stdout.splitlines():
    run_command([
        "docker", "exec", "hive-server", "beeline",
        "-u", "jdbc:hive2://localhost:10000/default",
        "--silent=true", "--showHeader=true", "--outputformat=tsv2",
        "-e", "DESCRIBE silver.sales_clean; SELECT COUNT(*) AS silver_rows FROM silver.sales_clean;",
    ])
else:
    print("silver.sales_clean is not registered yet; Checkpoint 4 owns that setup.")


## Checkpoint 3: Trace configuration, JDBC, schema, and Hive helpers

<details>
<summary>Open the source-reading guide</summary>

Target artifact: `/Users/richardwilkerson/IdeaProjects/HelloWorld/src/main/scala/WeekendHomework.scala`

Read the file in this order rather than top to bottom:

1. `main`: identify the Spark driver and environment variables.
2. `silverOutputSchema`: identify the clean contract before looking at transformations.
3. `spark.read.jdbc`: identify URL, credentials, driver, and source table responsibilities.
4. `runHiveQuery`: identify the HiveServer2 metadata path.
5. `readHiveTable`: identify the Parquet data path and empty-table behavior.
6. `finally`: identify temporary-directory and Spark-session cleanup ownership.

**Job connection:** Debugging follows boundaries. A failed JDBC read is not repaired by changing a `withColumn`, and a missing Hive table is not repaired by changing PostgreSQL credentials.

</details>


In [ ]:
source_text = scala_source.read_text(encoding="utf-8")
for marker in (
    "silverOutputSchema",
    "def runHiveQuery",
    "def readHiveTable",
    "def main",
    "spark.read.jdbc",
):
    line_number = source_text[: source_text.index(marker)].count("\n") + 1
    print(f"{marker}: line {line_number}")


## Checkpoint 4: Add idempotent clean-table setup and `--setup-only`

<details>
<summary>Open learner task, hints, and acceptance evidence</summary>

Target artifact: `/Users/richardwilkerson/IdeaProjects/HelloWorld/src/main/scala/WeekendHomework.scala`

Inside `main`, after loading the Hive settings and before reading `sales_clean`:

1. Build `/opt/hive/data/warehouse/silver.db/sales_clean` from `hiveWarehouseDirectory`.
2. Use the existing command helper to create that directory in `hive-server`.
3. Use `runHiveQuery` and `createExternalTableIfMissing` with `silverOutputSchema`.
4. After clean and rejected DataFrames have been read and counted—but before incremental selection—return when `args.contains("--setup-only")`.
5. Print a message that clearly states no transformations or appends ran.

Do not create the rejected table yet. Its final schema belongs to Part 2 because it must include diagnostic raw fields and `rejection_reason`.

**Hint:** Hive external-table DDL creates metadata that points to a directory; it does not create Parquet rows. `IF NOT EXISTS` is rerunnable but does not repair incompatible existing metadata, so compare the live `DESCRIBE` output with `silverOutputSchema`.

**Acceptance evidence:** The next cell prints `PASS: WH-M02 Part 1 setup contract is present`.

</details>


In [ ]:
part1_check = run_command([sys.executable, verify_script, "--mode", "part1"])
assert part1_check.returncode == 0, "Complete the focused Part 1 tasks above and rerun this cell"


## Checkpoint 5: Package and run setup-only mode

<details>
<summary>Open build, execution, and no-row-append instructions</summary>

Run `clean` and `package` in IntelliJ's sbt tool window. Confirm that the expected JAR exists before running the next cell.

The code cell records PostgreSQL, clean, and rejected counts; launches the packaged JAR with `--setup-only`; then records the counts again. Application arguments belong after the JAR path.

**Acceptance evidence:** Bronze, clean, and rejected counts are unchanged; the Scala output states that setup-only mode completed; no append-stage message appears.

**Job connection:** A successful process exit is weaker than a state invariant. Before/after observations prove that this run mode did not publish rows.

</details>


In [ ]:
def query_integer(command):
    result = run_command(command)
    assert result.returncode == 0
    values = [line.strip() for line in result.stdout.splitlines() if re.fullmatch(r"\d+", line.strip())]
    assert values, result.stdout
    return int(values[-1])

def postgres_count():
    return query_integer([
        "docker", "exec", "big-data-example-postgres",
        "psql", "-X", "-U", "bigdata", "-d", "week3_hive_migration",
        "-tAc", "SELECT COUNT(*) FROM bronze.sales_raw",
    ])

def hive_count(table, *, allow_missing=False):
    result = run_command([
        "docker", "exec", "hive-server", "beeline",
        "-u", "jdbc:hive2://localhost:10000/default",
        "--silent=true", "--showHeader=false", "--outputformat=tsv2",
        "-e", f"SELECT COUNT(*) FROM {table};",
    ])
    if result.returncode != 0 and allow_missing:
        print(f"{table} is not registered yet; treating its setup baseline as zero.")
        return 0
    assert result.returncode == 0
    values = [line.strip() for line in result.stdout.splitlines() if re.fullmatch(r"\d+", line.strip())]
    assert values, result.stdout
    return int(values[-1])

def state_counts():
    return {
        "bronze": postgres_count(),
        "clean": hive_count("silver.sales_clean"),
        "rejected": hive_count("silver.sales_rejected", allow_missing=True),
    }

assert spark_jar.is_file(), "Run clean and package in IntelliJ's sbt tool window first"
before_setup = state_counts()
jar_argument = shlex.quote(str(spark_jar))
setup_run = run_scala_shell(
    "spark-submit --class WeekendHomework --master 'local[*]' "
    "--packages org.postgresql:postgresql:42.7.7 "
    f"{jar_argument} --setup-only"
)
assert setup_run.returncode == 0
after_setup = state_counts()
assert after_setup == before_setup, f"Setup-only changed row counts: before={before_setup}, after={after_setup}"
print(f"PASS: setup-only preserved row counts: {after_setup}")


## Part 1 stop point

Stop here until all Part 1 checks pass and you can explain:

1. Why PostgreSQL uses the host's published port while Hive files require a container copy.
2. Why the Spark `StructType`, Hive table metadata, and Parquet files are separate contracts.
3. Which calls are lazy plans and which calls trigger Spark actions.
4. Why `finally` must remove staging files and stop Spark after success or failure.
5. What evidence proves that setup-only mode did not append rows.


# Part 2: Duplicates, cleaning, and transformations

Part 2 owns data changes. It selects only unprocessed Bronze rows, profiles duplicates, parses and standardizes raw values, assigns accepted or rejected outcomes, writes both outcomes, and proves that an unchanged rerun appends nothing.

## Data invariants

1. `bronze_row_id` is the persistent source identity and becomes `silver_row_id`.
2. A processed ID appears in clean or rejected Silver, never both.
3. Previously processed IDs do not re-enter the candidate set.
4. Candidate count equals accepted count plus rejected count.
5. Clean rows exactly match `silverOutputSchema`.
6. Invalid rows retain raw evidence and an explicit reason.
7. A rerun without new Bronze input leaves both Hive counts unchanged.


## Checkpoint 6: Define and inspect duplicates before removing them

<details>
<summary>Open duplicate categories, learner task, and hints</summary>

Target artifact: `/Users/richardwilkerson/IdeaProjects/HelloWorld/src/main/scala/WeekendHomework.scala`

Write the row-grain statement first. Then keep these problems separate:

| Duplicate type | Meaning | Owning check |
| --- | --- | --- |
| Source-ID duplicate | Candidate rows share `silver_row_id` | Group and count the current candidates |
| Replay duplicate | The ID is already accepted or rejected | Left anti join against the union of processed IDs |
| Exact duplicate | Every compared source value matches | Compare the deliberately selected raw columns |
| Possible business duplicate | Different IDs may describe the same sale line | Investigate a documented candidate business key |

An order number alone is not necessarily unique because one order may have several products or lines. `dropDuplicates` within one DataFrame does not provide cross-run idempotency; the processed-ID anti join does.

**Acceptance evidence:** Print candidate count and duplicate profiles before filtering. State which duplicate policy owns each result.

</details>


## Checkpoint 7: Parse, clean, validate, and reconcile

<details>
<summary>Open transformation contract and hints</summary>

Create parsed columns while retaining raw Bronze values. Normalize text with built-in Spark expressions; parse the order number and quantity as integers, the date with the expected format, and unit price as `decimal(12,2)`. Use `decimal`, not `double`, for currency.

Assign `rejection_reason` once after parsing. At minimum, handle missing/invalid dates, missing/invalid prices, nonpositive prices, missing/invalid quantities, nonpositive quantities, missing stable IDs, and any conflicting duplicate policy you chose.

Split accepted and rejected rows from the assessed DataFrame. Prove candidate count equals valid plus rejected and that their ID intersection is empty before writing.

For valid rows, calculate `sales_amount` as `decimal(14,2)` and select the exact clean schema in contract order. Rejected rows should retain raw diagnostic values plus their reason.

**Job connection:** Validation is a data-routing decision, not just a filter. Quarantine preserves evidence and prevents the same invalid source row from retrying forever.

</details>


## Checkpoint 8: Check my work — static Part 2 contract

<details>
<summary>Open validation scope</summary>

The next cell checks that the learner source contains the required incremental-selection, parsing, decimal, rejection, final-shaping, and two-output publication mechanisms. It does not prove runtime row correctness; the remaining checkpoints provide that evidence.

</details>


In [ ]:
part2_check = run_command([sys.executable, verify_script, "--mode", "part2"])
assert part2_check.returncode == 0, "Complete the focused Part 2 contract and rerun this cell"


## Checkpoint 9: Run the state-changing two-pass test

<details>
<summary>Open the state warning and execution sequence</summary>

**State warning:** This checkpoint inserts a five-row batch into the shared Week 3 Bronze table and appends its outcomes to Hive. It is intentionally not part of Part 1. Run it only when you want to advance the classroom data.

1. Save `WeekendHomework.scala`, then run `clean` and `package` in IntelliJ's sbt tool window.
2. Record baseline counts.
3. Run the supplied Python `generate_sales.py` once.
4. Run the Scala job without `--setup-only`.
5. Record first-run counts.
6. Run the same Scala job again without generating input.
7. Prove the second run did not change either Hive count.

The generated rows are intended to be valid, but the reconciliation uses accepted plus rejected so the contract remains correct when invalid input is introduced deliberately.

</details>


In [ ]:
assert spark_jar.is_file(), "Repackage the latest Scala source first"
before_batch = state_counts()
generator = shlex.quote(str(scala_project / "generate_sales.py"))
generate_result = run_scala_shell(
    f"{shlex.quote(str(project_root / '.venv/bin/python'))} {generator}"
)
assert generate_result.returncode == 0
after_generation = state_counts()
assert after_generation["bronze"] == before_batch["bronze"] + 5
assert after_generation["clean"] == before_batch["clean"]
assert after_generation["rejected"] == before_batch["rejected"]
print(f"Generated one batch: before={before_batch}, after={after_generation}")


In [ ]:
pipeline_command = (
    "spark-submit --class WeekendHomework --master 'local[*]' "
    "--packages org.postgresql:postgresql:42.7.7 "
    f"{shlex.quote(str(spark_jar))}"
)
first_run = run_scala_shell(pipeline_command)
assert first_run.returncode == 0
after_first_run = state_counts()
assert after_first_run["bronze"] == after_first_run["clean"] + after_first_run["rejected"]
print(f"First-run reconciliation: {after_first_run}")


In [ ]:
second_run = run_scala_shell(pipeline_command)
assert second_run.returncode == 0
after_second_run = state_counts()
assert after_second_run == after_first_run, (
    f"Unchanged rerun changed state: first={after_first_run}, second={after_second_run}"
)
print(f"PASS: unchanged rerun preserved counts: {after_second_run}")


## Checkpoint 10: Check my work — persisted runtime state

<details>
<summary>Open runtime validation contract</summary>

The runtime checker independently queries PostgreSQL and Hive. It requires Bronze count to equal clean plus rejected, zero duplicated IDs within each Hive table, and zero ID overlap between clean and rejected outcomes.

This check proves the currently visible state. The two-pass checkpoint above is what proves that this particular rerun added nothing.

</details>


In [ ]:
runtime_check = run_command([sys.executable, verify_script, "--mode", "runtime"])
assert runtime_check.returncode == 0


## Reflection

<details>
<summary>Open reflection questions</summary>

1. Why can `dropDuplicates("silver_row_id")` help within one candidate DataFrame but not protect an unchanged rerun by itself?
2. Why must rejected IDs be included in the next run's processed-ID set?
3. Where can this file-copy publication fail after data becomes partially visible?
4. Why does `IF NOT EXISTS` fail to repair incompatible existing Hive metadata?
5. Which actions cause repeated JDBC or Parquet scans, and when would caching be justified?
6. How would a transactional table format change the file-copy and metadata publication boundary?

</details>


## Finish, restart, and state boundaries

<details>
<summary>Open the restart and reset guide</summary>

- Restart the notebook kernel: clears Python variables only; PostgreSQL, Hive, files, and containers remain.
- Clear notebook output: changes presentation only.
- Stop Spark: each submitted application stops its own Spark session in `finally`.
- Stop services without deleting state: `docker stop hive-server` and the repository's PostgreSQL Compose `stop` command.
- Restore learner source: because `HelloWorld` is an external project without a Git repository, make a deliberate backup before major edits; this notebook does not offer a destructive automatic restore.
- Reset generated rows: no automatic delete is supplied because this exercise shares the Week 3 migration tables and Hive warehouse. Do not manually delete source rows or Parquet files unless you are intentionally rebuilding the entire fixture.

The missing isolated reset is a known limitation of this draft lab and an example of why production-quality exercises should own isolated state.

</details>

## Definition of done

- Part 1 static and setup-only checks pass without changing row counts.
- You can explain the PostgreSQL, Mac/Spark, Parquet, Hive filesystem, and Hive metastore boundaries.
- Part 2 static check passes and the candidate split reconciles before publication.
- The first state-changing run processes the generated batch.
- The second unchanged run leaves clean and rejected counts unchanged.
- Runtime verification reports zero duplicate IDs, zero clean/rejected overlap, and full Bronze reconciliation.
- You can explain why the classroom append process is not atomic and what stronger publication mechanism a production design would need.
